In [17]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(ggpubr)
    library(BSgenome)
    library(biomaRt)
    library(BSgenome.Ocuniculus.NCBI.oryCun2)
    library(GenomicFeatures)
    library(compEpiTools)
    library(GenomeInfoDb)
    library(stringi)
    library(AnnotationHub)
})

In [18]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_2'
io$output.directory <- file.path(io$basedir,"ArchR")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

In [19]:
opts = list()
# Options
opts$min.fragments <- 2500
opts$filterTSS.score <- 2 # May need to rerun with threshold at 2

# ArchR options
addArchRThreads(threads = 1) 

Setting default number of Parallel threads to 1.



In [20]:
# Create genome annotation
genomeAnnotation <- createGenomeAnnotation(genome = BSgenome.Ocuniculus.NCBI.oryCun2)

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [21]:
# Create a gene annotation using Txdb file
rabbit_txdb <- makeTxDbFromBiomart(biomart="ensembl", dataset="ocuniculus_gene_ensembl")

Warning message:
"Ensembl will soon enforce the use of https.
Ensure the 'host' argument includes "https://""
Download and preprocess the 'transcripts' data frame ... 
OK

Download and preprocess the 'chrominfo' data frame ... 
OK

Download and preprocess the 'splicings' data frame ... 
OK

Download and preprocess the 'genes' data frame ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
OK



In [22]:
taxid = taxonomyId(rabbit_txdb)
## If we don't have a package, then lets get the taxIds and AHIds
        ## for the hub objects
        loadNamespace("AnnotationHub")
        ah <- AnnotationHub::AnnotationHub()
        ah <- subset(ah, ah$rdataclass=='OrgDb') 
        mc <- mcols(ah)[,'taxonomyid', drop=FALSE]
        ## Then just get the object
        AHID <- rownames(mc[mc$taxonomyid==taxid,,drop=FALSE])
        rabbit_OrgDb <- ah[[AHID[2]]]

<environment: namespace:AnnotationHub>

snapshotDate(): 2021-10-20

loading from cache



In [23]:
#rabbit_txdb
geneAnnotation <- createGeneAnnotation(
  TxDb  = rabbit_txdb,
    OrgDb =rabbit_OrgDb,
  TSS = TSS(rabbit_txdb), 
  exons = exons(rabbit_txdb)
)

'select()' returned 1:1 mapping between keys and columns

Getting Genes..

Determined Annotation Style = ENSEMBL

Getting Exons..

Getting TSS..



In [24]:
# Put genome in right style
seqstyle = 'UCSC'
#seqstyle = 'NCBI'
seqlevelsStyle(genomeAnnotation$chromSizes) <- seqstyle
seqlevelsStyle(geneAnnotation$genes) <- seqstyle
seqlevelsStyle(geneAnnotation$TSS) <- seqstyle
seqlevelsStyle(geneAnnotation$exons) <- seqstyle
seqlevelsStyle(BSgenome.Ocuniculus.NCBI.oryCun2) <- seqstyle

#important as rabbit chromosome dont have the 'chr' prefix
addArchRChrPrefix(chrPrefix = FALSE)

ArchR is now disabling the requirement of chromosome prefix = 'chr'



In [34]:
fragment_files = list.files(file.path(io$basedir, 'data/'))

In [35]:
fragment_files

[1] "BGRGP1_fragments.tsv.gz" "BGRGP2_fragments.tsv.gz"
[3] "BGRGP3_fragments.tsv.gz" "BGRGP4_fragments.tsv.gz"
[5] "BGRGP5_fragments.tsv.gz" "BGRGP6_fragments.tsv.gz"
[7] "BGRGP7_fragments.tsv.gz" "BGRGP8_fragments.tsv.gz"

In [36]:
#Create Arrow File for filtered samples, filtered using Signac's filter for ATAC
ArrowFiles <- createArrowFiles(
  inputFiles = file.path(io$basedir, 'data', fragment_files),
  sampleNames = paste0('rabbit_', strsplit(fragment_files,"_") %>% map_chr(1)),
  minTSS = opts$filterTSS.score, #Dont set this too high because you can always increase later
  minFrags = opts$min.fragments , 
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

Found Gene Seqnames not in GenomeAnnotation chromSizes, Removing : chrM,AAGW02076170,AAGW02077230,AAGW02077833,AAGW02078230,AAGW02078537,AAGW02078775,AAGW02078982,AAGW02079106,AAGW02079147,AAGW02079168,AAGW02079214,AAGW02079242,AAGW02079307,AAGW02079327,AAGW02079371,AAGW02079515,AAGW02079587,AAGW02079588,AAGW02079657,AAGW02079658,AAGW02079674,AAGW02079688,AAGW02079913,AAGW02079997,AAGW02080057,AAGW02080074,AAGW02080085,AAGW02080103,AAGW02080168,AAGW02080217,AAGW02080234,AAGW02080240,AAGW02080326,AAGW02080430,AAGW02080442,AAGW02080504,AAGW02080517,AAGW02080613,AAGW02080699,AAGW02080733,AAGW02080735,AAGW02080770,AAGW02080791,AAGW02080820,AAGW02080821,AAGW02080938,AAGW02080941,AAGW02080961,AAGW02080967,AAGW02080980,AAGW02081012,AAGW02081119,AAGW02081122,AAGW02081136,AAGW02081154,AAGW02081155,AAGW02081156,AAGW02081157,AAGW02081158,AAGW02081159,AAGW02081160,AAGW02081161,AAGW02081169,AAGW02081209,AAGW02081222,AAGW02081234,AAGW02081239,AAGW02081247,AAGW02081253,AAGW02081303,AAGW02081305,AAGW0

Found Exon Seqnames not in GenomeAnnotation chromSizes, Removing : chrM,AAGW02076170,AAGW02077230,AAGW02077833,AAGW02078230,AAGW02078537,AAGW02078775,AAGW02078982,AAGW02079106,AAGW02079147,AAGW02079168,AAGW02079214,AAGW02079242,AAGW02079307,AAGW02079327,AAGW02079371,AAGW02079515,AAGW02079587,AAGW02079588,AAGW02079657,AAGW02079658,AAGW02079674,AAGW02079688,AAGW02079913,AAGW02079997,AAGW02080057,AAGW02080074,AAGW02080085,AAGW02080103,AAGW02080168,AAGW02080217,AAGW02080234,AAGW02080240,AAGW02080326,AAGW02080430,AAGW02080442,AAGW02080504,AAGW02080517,AAGW02080613,AAGW02080699,AAGW02080733,AAGW02080735,AAGW02080770,AAGW02080791,AAGW02080820,AAGW02080821,AAGW02080938,AAGW02080941,AAGW02080961,AAGW02080967,AAGW02080980,AAGW02081012,AAGW02081119,AAGW02081122,AAGW02081136,AAGW02081154,AAGW02081155,AAGW02081156,AAGW02081157,AAGW02081158,AAGW02081159,AAGW02081160,AAGW02081161,AAGW02081169,AAGW02081209,AAGW02081222,AAGW02081234,AAGW02081239,AAGW02081247,AAGW02081253,AAGW02081303,AAGW02081305,AAGW0

Found TSS Seqnames not in GenomeAnnotation chromSizes, Removing : chrM,AAGW02076170,AAGW02077230,AAGW02077833,AAGW02078230,AAGW02078537,AAGW02078775,AAGW02078982,AAGW02079106,AAGW02079147,AAGW02079168,AAGW02079214,AAGW02079242,AAGW02079307,AAGW02079327,AAGW02079371,AAGW02079515,AAGW02079587,AAGW02079588,AAGW02079657,AAGW02079658,AAGW02079674,AAGW02079688,AAGW02079913,AAGW02079997,AAGW02080057,AAGW02080074,AAGW02080085,AAGW02080103,AAGW02080168,AAGW02080217,AAGW02080234,AAGW02080240,AAGW02080326,AAGW02080430,AAGW02080442,AAGW02080504,AAGW02080517,AAGW02080613,AAGW02080699,AAGW02080733,AAGW02080735,AAGW02080770,AAGW02080791,AAGW02080820,AAGW02080821,AAGW02080938,AAGW02080941,AAGW02080961,AAGW02080967,AAGW02080980,AAGW02081012,AAGW02081119,AAGW02081122,AAGW02081136,AAGW02081154,AAGW02081155,AAGW02081156,AAGW02081157,AAGW02081158,AAGW02081159,AAGW02081160,AAGW02081161,AAGW02081169,AAGW02081209,AAGW02081222,AAGW02081234,AAGW02081239,AAGW02081247,AAGW02081253,AAGW02081303,AAGW02081305,AAGW02

ArchR logging to : ArchRLogs/ArchR-createArrows-ee31925782c94-Date-2021-12-04_Time-15-04-07.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2021-12-04 15:04:08 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP6 : 1 of 8) Determining Arrow Method to use!

Attempting to index /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_2/data/BGRGP6_fragments.tsv.gz as tabix..

2021-12-04 15:06:00 : (rabbit_BGRGP6 : 1 of 8) Reading In Fragments from inputFiles (readMethod = tabix), 1.872 mins elapsed.

2021-12-04 15:06:00 : (rabbit_BGRGP6 : 1 of 8) Tabix Bed To Temporary File, 1.873 mins elapsed.

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr1:1-38970151

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr1:38970152-77940302

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr1:77940303-116910454

(rabbit_BGRGP6 : 1 of 8) found fragments when removed ch

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr20:13276533-19914799

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr20:19914800-26553065

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr20:26553066-33191332

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr21:1-3115655

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr21:3115656-6231310

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr21:6231311-9346965

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome prefix : chr21:9346966-12462620

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
"one argument not used by format '%s Reading TabixFile %s Percent'"
2021-12-04 15:22:16 : (rabbit_BGRGP6 : 1 of 8) Reading TabixFile 64 Percent, 18.149 mins elapsed.

(rabbit_BGRGP6 : 1 of 8) found fragments when removed chromosome pref

...

2021-12-04 19:40:23 : (rabbit_BGRGP5 : 7 of 8) Tabix Bed To Temporary File, 276.252 mins elapsed.

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr1:1-38970151

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr1:38970152-77940302

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr1:77940303-116910454

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr1:116910455-155880605

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr1:155880606-194850757

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr10:1-9599448

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr10:9599449-19198896

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr10:19198897-28798344

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr10:28798345-38397792

Warning message in sprintf

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr21:9346966-12462620

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
"one argument not used by format '%s Reading TabixFile %s Percent'"
2021-12-04 19:54:36 : (rabbit_BGRGP5 : 7 of 8) Reading TabixFile 64 Percent, 290.48 mins elapsed.

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr21:12462621-15578276

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr3:1-31138221

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr3:31138222-62276442

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr3:62276443-93414663

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr3:93414664-124552884

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome prefix : chr3:124552885-155691105

(rabbit_BGRGP5 : 7 of 8) found fragments when removed chromosome p

2021-12-04 20:13:24 : (rabbit_BGRGP5 : 7 of 8) Removing Fragments from Filtered Cells, 309.273 mins elapsed.

2021-12-04 20:13:24 : (rabbit_BGRGP5 : 7 of 8) Creating Filtered Arrow File, 309.273 mins elapsed.

2021-12-04 20:14:57 : (rabbit_BGRGP5 : 7 of 8) Finished Constructing Filtered Arrow File!, 310.818 mins elapsed.

2021-12-04 20:14:57 : (rabbit_BGRGP5 : 7 of 8) Adding TileMatrix!, 310.82 mins elapsed.

2021-12-04 20:18:33 : (rabbit_BGRGP5 : 7 of 8) Adding GeneScoreMatrix!, 314.416 mins elapsed.

2021-12-04 20:23:54 : (rabbit_BGRGP5 : 7 of 8) Finished Creating Arrow File, 319.781 mins elapsed.

(rabbit_BGRGP4 : 8 of 8) Determining Arrow Method to use!

Attempting to index /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_2/data/BGRGP4_fragments.tsv.gz as tabix..

2021-12-04 20:25:17 : (rabbit_BGRGP4 : 8 of 8) Reading In Fragments from inputFiles (readMethod = tabix), 321.162 mins elapsed.

2021-12-04 20:25:17 : (rabbit_BGRGP4 : 8 of 8) Tabix Bed To Temporary File, 321.163 m

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr2:139465850-174332312

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr20:1-6638266

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr20:6638267-13276532

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr20:13276533-19914799

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr20:19914800-26553065

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr20:26553066-33191332

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr21:1-3115655

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr21:3115656-6231310

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr21:6231311-9346965

(rabbit_BGRGP4 : 8 of 8) found fragments when removed chromosome prefix : chr21:9346966-12462620

Warning message in sprintf("%s 

2021-12-04 20:52:00 : (rabbit_BGRGP4 : 8 of 8) CellStats : Median TSS Enrichment = 2.9385 , 347.879 mins elapsed.


2021-12-04 20:52:03 : (rabbit_BGRGP4 : 8 of 8) Adding Additional Feature Counts!, 347.919 mins elapsed.

Warning message in valid.GenomicRanges.seqinfo(x, suggest.trim = TRUE):
"GRanges object contains 3 out-of-bound ranges located on sequences
  chr12, chr13, and chr16. Note that ranges located on a sequence whose
  length is unknown (NA) or on a circular sequence are not considered
  out-of-bound (use seqlengths() and isCircular() to get the lengths and
  circularity flags of the underlying sequences). You can use trim() to
  trim these ranges. See ?`trim,GenomicRanges-method` for more
  information."
Warning message in valid.GenomicRanges.seqinfo(x, suggest.trim = TRUE):
"GRanges object contains 3 out-of-bound ranges located on sequences
  chr12, chr13, and chr16. Note that ranges located on a sequence whose
  length is unknown (NA) or on a circular sequence are not co

In [ ]:
ArrowFiles = list.files(io$output.directory, pattern ='arrow')

In [ ]:
# Calculate doublet scores
doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 10, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)

In [ ]:
proj <- ArchRProject(
  ArrowFiles = ArrowFiles, 
  outputDirectory = "Project",
  copyArrows = TRUE, #This is recommened so that you maintain an unaltered copy for later usage.
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

In [ ]:
proj <- filterDoublets(ArchRProj = proj)

In [ ]:
proj <- saveArchRProject(ArchRProj = proj)